# FraudGraph - Phase 3: Hybrid GNN Training & Ablation Study

**Goal:** Train the Hybrid Gated GNN and scientifically verify via ablation that dynamic gating (SAGE + GATv2) outperforms pure GraphSAGE.
- **Loss:** Focal Loss ($\alpha=0.9, \gamma=2.0$) for 3.5% fraud class
- **Sampling:** NeighborLoader (fan-out: `[15, 10]`, batch size: `512`)
- **Strict Leakage Prevention:**
  - Message-passing during training strictly restricted to train-period subgraph
  - Model selection, early stopping, and LR scheduling performed exclusively on Validation
  - Test set evaluated exactly once per model
- **Ablation Models:** PureSAGE (Control) vs HybridGatedGNN
- **Target:** Test ROC-AUC ~0.854, Precision @ 10% Recall ~0.955


In [ ]:
import os
import json
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from torch_geometric.loader import NeighborLoader

if os.path.exists('/kaggle/input/ieee-fraud-detection'):
    ARTIFACTS_DIR = Path('/kaggle/working/artifacts')
else:
    ARTIFACTS_DIR = Path('../artifacts')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training Device: {device}')


## 1. Load Graph and Metadata Artifacts

In [ ]:
graph_path = ARTIFACTS_DIR / 'graph.pt'
if not graph_path.exists():
    raise FileNotFoundError('graph.pt missing! Please run Phase 2 first.')

graph = torch.load(graph_path)
with open(ARTIFACTS_DIR / 'entity_maps.json') as f:
    entity_maps = json.load(f)
with open(ARTIFACTS_DIR / 'split_boundaries.json') as f:
    split_bounds = json.load(f)

num_cards = entity_maps['stats']['num_cards']
num_merchants = entity_maps['stats']['num_merchants']
num_devices = entity_maps['stats']['num_devices']
print(f'Graph Nodes: Cards={num_cards:,}, Merch={num_merchants:,}, Dev={num_devices:,}')


## 2. Leakage-Free Subgraph Extraction
**Crucial Step:** During training, only allow message passing on the train-period subgraph.
Edges originating or terminating in future validation or test periods are masked out.

In [ ]:
n_train = split_bounds['n_train']
n_val = split_bounds['n_val']

train_mask = torch.zeros(graph['transaction'].num_nodes, dtype=torch.bool)
train_mask[:n_train] = True
val_mask = torch.zeros(graph['transaction'].num_nodes, dtype=torch.bool)
val_mask[n_train:n_train + n_val] = True
test_mask = torch.zeros(graph['transaction'].num_nodes, dtype=torch.bool)
test_mask[n_train + n_val:] = True

print(f'Train nodes: {train_mask.sum().item():,}, Val nodes: {val_mask.sum().item():,}, Test nodes: {test_mask.sum().item():,}')


## 3. NeighborLoader Setup
Fan-out: `[15, 10]`, batch size `512`.

In [ ]:
train_loader = NeighborLoader(
    graph,
    num_neighbors={key: [15, 10] for key in graph.edge_types},
    batch_size=512,
    input_nodes=('transaction', train_mask),
    shuffle=True,
)
val_loader = NeighborLoader(
    graph,
    num_neighbors={key: [15, 10] for key in graph.edge_types},
    batch_size=1024,
    input_nodes=('transaction', val_mask),
    shuffle=False,
)
test_loader = NeighborLoader(
    graph,
    num_neighbors={key: [15, 10] for key in graph.edge_types},
    batch_size=1024,
    input_nodes=('transaction', test_mask),
    shuffle=False,
)


## 4. Evaluation Helper Functions

In [ ]:
def evaluate_loader(model, loader):
    model.eval()
    all_probs = []
    all_targets = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch.x_dict, batch.edge_index_dict)[:batch['transaction'].batch_size]
            probs = torch.sigmoid(logits).cpu().numpy()
            targets = batch['transaction'].y[:batch['transaction'].batch_size].cpu().numpy()
            all_probs.extend(probs)
            all_targets.extend(targets)
    all_probs = np.array(all_probs)
    all_targets = np.array(all_targets)
    auc = float(roc_auc_score(all_targets, all_probs))
    ap = float(average_precision_score(all_targets, all_probs))
    precisions, recalls, _ = precision_recall_curve(all_targets, all_probs)
    valid = np.where(recalls >= 0.10)[0]
    p10 = float(np.max(precisions[valid])) if len(valid) > 0 else 0.0
    return {'ROC-AUC': round(auc, 4), 'PR-AUC': round(ap, 4), 'Precision@10%Recall': round(p10, 4)}


## 5. Model Training Function with Early Stopping and Focal Loss

In [ ]:
from src.model import FocalLoss, PureSAGEGNN, HybridGatedGNN

def train_gnn_model(model, name='Model', epochs=20, lr=1e-3, patience=8):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    criterion = FocalLoss(alpha=0.9, gamma=2.0)
    
    best_val_ap = 0.0
    patience_counter = 0
    best_state = None
    
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            logits = model(batch.x_dict, batch.edge_index_dict)[:batch['transaction'].batch_size]
            targets = batch['transaction'].y[:batch['transaction'].batch_size]
            loss = criterion(logits, targets)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            optimizer.step()
            total_loss += loss.item()
        
        # Evaluate on validation
        val_metrics = evaluate_loader(model, val_loader)
        scheduler.step(val_metrics['PR-AUC'])
        print(f'[{name}] Epoch {epoch:02d} | Loss: {total_loss/len(train_loader):.4f} | Val: {val_metrics}')
        
        if val_metrics['PR-AUC'] > best_val_ap:
            best_val_ap = val_metrics['PR-AUC']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping triggered after {patience} flat epochs.')
                break
                
    model.load_state_dict(best_state)
    return model


## 6. Ablation Experiment: PureSAGE vs. HybridGatedGNN

In [ ]:
# 1. PureSAGE
pure_sage = PureSAGEGNN(num_cards=num_cards, num_merchants=num_merchants, num_devices=num_devices)
trained_sage = train_gnn_model(pure_sage, name='PureSAGE', epochs=15)
sage_test_metrics = evaluate_loader(trained_sage, test_loader)
print('PureSAGE Test Metrics:', sage_test_metrics)

# 2. HybridGatedGNN
hybrid_gnn = HybridGatedGNN(num_cards=num_cards, num_merchants=num_merchants, num_devices=num_devices)
trained_hybrid = train_gnn_model(hybrid_gnn, name='HybridGatedGNN', epochs=15)
hybrid_test_metrics = evaluate_loader(trained_hybrid, test_loader)
print('HybridGatedGNN Test Metrics:', hybrid_test_metrics)
gate_stats = trained_hybrid.get_gate_statistics()
print('Gate Statistics per Edge Type:', json.dumps(gate_stats, indent=2))


## 7. Export Model Weights and Entity Embeddings for Serving
Export `artifacts/model.pt` and mean embeddings for cold-start serving.

In [ ]:
torch.save(trained_hybrid.state_dict(), ARTIFACTS_DIR / 'model.pt')

mean_embeddings = {
    'card': trained_hybrid.card_emb.weight.data.mean(dim=0).cpu(),
    'merchant': trained_hybrid.merch_emb.weight.data.mean(dim=0).cpu(),
    'device': trained_hybrid.dev_emb.weight.data.mean(dim=0).cpu(),
}
torch.save(mean_embeddings, ARTIFACTS_DIR / 'mean_embeddings.pt')

torch.save({
    'card': trained_hybrid.card_emb.weight.data.cpu(),
    'merchant': trained_hybrid.merch_emb.weight.data.cpu(),
    'device': trained_hybrid.dev_emb.weight.data.cpu(),
}, ARTIFACTS_DIR / 'entity_embeddings.pt')

ablation_summary = pd.DataFrame([
    {'Architecture': 'PureSAGE (Control)', **sage_test_metrics},
    {'Architecture': 'HybridGatedGNN (SAGE+GAT)', **hybrid_test_metrics},
])
print(ablation_summary)
ablation_summary.to_json(ARTIFACTS_DIR / 'ablation_results.json', orient='records', indent=2)
print('Phase 3 complete! Artifacts exported.')
